# 臨床試驗假設檢定

> **課程定位**：基本工具篇（2/5）  
> **前置課程**：[01_醫療統計基礎框架](./01_醫療統計基礎框架.ipynb)  
> **學習路徑**：01 基礎框架 → **02 臨床試驗** → 03 流行病學 → 04 產後憂鬱案例 → 05 醫療品質

## 學習目標
- 模擬並分析完整的隨機對照試驗（RCT）數據
- 建立臨床論文標準的「Table 1」基線特徵比較
- 應用 t 檢定、配對 t 檢定、卡方檢定於臨床指標
- 執行多組比較（ANOVA）進行劑量反應分析
- 理解並實作非劣性檢定（TOST）

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Microsoft JhengHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

## 1. 情境設定：第二型糖尿病降血糖藥物臨床試驗

### 試驗設計
- **目的**：評估新藥 GlucoDown 對第二型糖尿病患者的降血糖效果
- **設計**：隨機、雙盲、安慰劑對照試驗
- **受試者**：200 名第二型糖尿病患者（1:1 隨機分配）
- **主要指標**：12 週後空腹血糖（FPG）變化量
- **次要指標**：達到目標血糖（FPG < 7.0 mmol/L）的比例

In [ ]:
np.random.seed(42)
n_total = 200

# 基本人口學
patient_ids = [f'PT-{i:04d}' for i in range(1, n_total + 1)]
groups = np.array(['治療組'] * 100 + ['安慰劑組'] * 100)
np.random.shuffle(groups)

ages = np.random.normal(55, 10, n_total).astype(int)
ages = np.clip(ages, 30, 80)
sex = np.random.choice(['男', '女'], n_total, p=[0.55, 0.45])
bmi = np.random.normal(28, 4, n_total).round(1)
diabetes_years = np.random.exponential(5, n_total).round(1)

# 基線血糖
baseline_fpg = np.random.normal(9.5, 1.5, n_total).round(1)
baseline_fpg = np.clip(baseline_fpg, 7.0, 15.0)

# 12 週後血糖
post_fpg = np.zeros(n_total)
for i in range(n_total):
    if groups[i] == '治療組':
        # 新藥效果：平均降 2.0 mmol/L
        change = np.random.normal(-2.0, 1.2)
    else:
        # 安慰劑效果：平均降 0.5 mmol/L
        change = np.random.normal(-0.5, 1.2)
    post_fpg[i] = round(baseline_fpg[i] + change, 1)
    post_fpg[i] = max(post_fpg[i], 4.0)

fpg_change = np.round(post_fpg - baseline_fpg, 1)

# 是否達標
target_achieved = (post_fpg < 7.0).astype(int)

# 合併高血壓
hypertension = np.random.choice([0, 1], n_total, p=[0.6, 0.4])

# 建立 DataFrame
df = pd.DataFrame({
    'patient_id': patient_ids,
    'group': groups,
    'age': ages,
    'sex': sex,
    'bmi': bmi,
    'diabetes_years': diabetes_years,
    'hypertension': hypertension,
    'baseline_fpg': baseline_fpg,
    'post_fpg': post_fpg,
    'fpg_change': fpg_change,
    'target_achieved': target_achieved
})

print("臨床試驗數據總覽")
print(f"總受試者: {n_total} 人")
print(f"治療組: {(groups=='治療組').sum()} 人, 安慰劑組: {(groups=='安慰劑組').sum()} 人")
display(df.head(10))

## 2. 基線特徵比較（Table 1）

臨床論文的第一張表格，用於驗證隨機分配是否成功——兩組的基線特徵應該相近。

- **連續變數**：用獨立樣本 t 檢定
- **類別變數**：用卡方檢定

In [ ]:
def create_table1(df, group_col='group'):
    """建立臨床論文標準 Table 1"""
    groups = df[group_col].unique()
    g1, g2 = groups[0], groups[1]
    df1 = df[df[group_col] == g1]
    df2 = df[df[group_col] == g2]
    
    rows = []
    
    # 樣本數
    rows.append({'變數': 'N', g1: str(len(df1)), g2: str(len(df2)), 'p-value': ''})
    
    # 連續變數
    continuous_vars = {
        'age': '年齡 (歲)',
        'bmi': 'BMI (kg/m²)',
        'diabetes_years': '糖尿病病程 (年)',
        'baseline_fpg': '基線空腹血糖 (mmol/L)'
    }
    
    for var, label in continuous_vars.items():
        m1, s1 = df1[var].mean(), df1[var].std()
        m2, s2 = df2[var].mean(), df2[var].std()
        t_stat, p_val = stats.ttest_ind(df1[var], df2[var])
        rows.append({
            '變數': label,
            g1: f'{m1:.1f} \u00b1 {s1:.1f}',
            g2: f'{m2:.1f} \u00b1 {s2:.1f}',
            'p-value': f'{p_val:.3f}'
        })
    
    # 類別變數
    # 性別
    male1 = (df1['sex'] == '男').sum()
    male2 = (df2['sex'] == '男').sum()
    contingency = pd.crosstab(df[group_col], df['sex'])
    chi2, p_val, _, _ = stats.chi2_contingency(contingency)
    rows.append({
        '變數': '男性, n (%)',
        g1: f'{male1} ({male1/len(df1)*100:.1f}%)',
        g2: f'{male2} ({male2/len(df2)*100:.1f}%)',
        'p-value': f'{p_val:.3f}'
    })
    
    # 高血壓
    ht1 = df1['hypertension'].sum()
    ht2 = df2['hypertension'].sum()
    contingency_ht = pd.crosstab(df[group_col], df['hypertension'])
    chi2, p_val, _, _ = stats.chi2_contingency(contingency_ht)
    rows.append({
        '變數': '合併高血壓, n (%)',
        g1: f'{ht1} ({ht1/len(df1)*100:.1f}%)',
        g2: f'{ht2} ({ht2/len(df2)*100:.1f}%)',
        'p-value': f'{p_val:.3f}'
    })
    
    return pd.DataFrame(rows)

table1 = create_table1(df)
print("=" * 70)
print("Table 1: 基線特徵比較")
print("=" * 70)
display(table1)
print("\n結論：所有基線變數的 p-value 均 > 0.05 → 隨機分配成功，兩組具有可比性")

## 3. 主要療效指標：雙樣本 t 檢定

**研究假設：**
- H\u2080: 治療組與安慰劑組的血糖變化量無差異
- H\u2081: 治療組的血糖下降量大於安慰劑組

In [ ]:
treatment = df[df['group'] == '治療組']['fpg_change']
placebo = df[df['group'] == '安慰劑組']['fpg_change']

# 假設檢驗
# Step 1: 常態性檢驗
stat_t, p_norm_t = stats.shapiro(treatment)
stat_p, p_norm_p = stats.shapiro(placebo)

# Step 2: 變異數齊性檢驗
stat_lev, p_levene = stats.levene(treatment, placebo)

# Step 3: t 檢定
t_stat, p_value = stats.ttest_ind(treatment, placebo)

# 效應量
mean_diff = treatment.mean() - placebo.mean()
sp = np.sqrt(((len(treatment)-1)*treatment.std(ddof=1)**2 + 
              (len(placebo)-1)*placebo.std(ddof=1)**2) / (len(treatment)+len(placebo)-2))
cohens_d = mean_diff / sp

# 95% CI
se = sp * np.sqrt(1/len(treatment) + 1/len(placebo))
ci_lower = mean_diff - 1.96 * se
ci_upper = mean_diff + 1.96 * se

print("=" * 60)
print("主要療效指標分析：空腹血糖變化量（FPG Change）")
print("=" * 60)

print(f"\n假設檢驗前置：")
print(f"  Shapiro-Wilk (治療組): p = {p_norm_t:.4f} {'常態' if p_norm_t > 0.05 else '非常態'}")
print(f"  Shapiro-Wilk (安慰劑): p = {p_norm_p:.4f} {'常態' if p_norm_p > 0.05 else '非常態'}")
print(f"  Levene's test:         p = {p_levene:.4f} {'變異數齊性' if p_levene > 0.05 else '變異數不齊'}")

print(f"\n描述性統計：")
print(f"  治療組: {treatment.mean():.2f} \u00b1 {treatment.std():.2f} mmol/L (n={len(treatment)})")
print(f"  安慰劑: {placebo.mean():.2f} \u00b1 {placebo.std():.2f} mmol/L (n={len(placebo)})")

print(f"\n分析結果：")
print(f"  平均差異: {mean_diff:.2f} mmol/L")
print(f"  95% CI:   ({ci_lower:.2f}, {ci_upper:.2f})")
print(f"  t 統計量: {t_stat:.3f}")
print(f"  p-value:  {p_value:.6f} {'***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'}")
print(f"  Cohen's d: {cohens_d:.3f} ({'小' if abs(cohens_d)<0.5 else '中' if abs(cohens_d)<0.8 else '大'}效應)")

# 視覺化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].boxplot([treatment, placebo], labels=['治療組', '安慰劑組'], patch_artist=True,
                boxprops=dict(facecolor='lightblue'), medianprops=dict(color='red', linewidth=2))
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_ylabel('血糖變化量 (mmol/L)')
axes[0].set_title('主要指標：血糖變化量比較', fontsize=13, fontweight='bold')

axes[1].hist(treatment, bins=15, alpha=0.6, label='治療組', color='blue', edgecolor='white')
axes[1].hist(placebo, bins=15, alpha=0.6, label='安慰劑組', color='red', edgecolor='white')
axes[1].set_xlabel('血糖變化量 (mmol/L)')
axes[1].set_ylabel('人數')
axes[1].set_title('血糖變化量分布', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. 配對 t 檢定：治療組的前後比較

檢驗治療組在治療前後的血糖是否有顯著改善。

In [ ]:
df_treatment = df[df['group'] == '治療組']
baseline = df_treatment['baseline_fpg']
post = df_treatment['post_fpg']

t_paired, p_paired = stats.ttest_rel(baseline, post)

print("=" * 50)
print("配對 t 檢定：治療組前後比較")
print("=" * 50)
print(f"\n基線 FPG: {baseline.mean():.2f} \u00b1 {baseline.std():.2f} mmol/L")
print(f"治療後 FPG: {post.mean():.2f} \u00b1 {post.std():.2f} mmol/L")
print(f"平均變化: {(post-baseline).mean():.2f} mmol/L")
print(f"t = {t_paired:.3f}, p = {p_paired:.6f}")

# Paired line plot (sample of 30 patients)
fig, ax = plt.subplots(figsize=(12, 6))

sample_idx = df_treatment.sample(30, random_state=42).index
for idx in sample_idx:
    color = 'green' if df.loc[idx, 'post_fpg'] < df.loc[idx, 'baseline_fpg'] else 'red'
    ax.plot([0, 1], [df.loc[idx, 'baseline_fpg'], df.loc[idx, 'post_fpg']], 
            color=color, alpha=0.4, linewidth=1)

ax.plot([0, 1], [baseline.mean(), post.mean()], 'k-o', linewidth=3, markersize=10, label='平均值')
ax.set_xticks([0, 1])
ax.set_xticklabels(['基線', '12 週後'], fontsize=14)
ax.set_ylabel('空腹血糖 (mmol/L)', fontsize=12)
ax.set_title('治療組個別患者血糖變化軌跡（抽樣 30 人）', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. 次要指標：卡方檢定（達標率比較）

**次要指標**：12 週後達到目標血糖（FPG < 7.0 mmol/L）的比例。

In [ ]:
# 列聯表
contingency = pd.crosstab(df['group'], df['target_achieved'], margins=True)
contingency.columns = ['未達標', '達標', '合計']
contingency.index = [contingency.index[0], contingency.index[1], '合計']

print("=" * 50)
print("次要指標：達標率比較（FPG < 7.0 mmol/L）")
print("=" * 50)
display(contingency)

# 卡方檢定
ct = pd.crosstab(df['group'], df['target_achieved'])
chi2, p_chi, dof, expected = stats.chi2_contingency(ct)

# 達標率
rate_t = df[df['group']=='治療組']['target_achieved'].mean()
rate_p = df[df['group']=='安慰劑組']['target_achieved'].mean()

# RR, OR, NNT
a = ct.loc['治療組', 1]
b = ct.loc['治療組', 0]
c = ct.loc['安慰劑組', 1]
d = ct.loc['安慰劑組', 0]

RR = (a/(a+b)) / (c/(c+d))
OR = (a*d) / (b*c)
ARR = rate_t - rate_p
NNT = 1/ARR if ARR != 0 else float('inf')

print(f"\n治療組達標率: {rate_t:.1%}")
print(f"安慰劑達標率: {rate_p:.1%}")
print(f"\n\u03c7\u00b2 = {chi2:.3f}, df = {dof}, p = {p_chi:.4f}")
print(f"RR  = {RR:.3f}")
print(f"OR  = {OR:.3f}")
print(f"ARR = {ARR:.1%}")
print(f"NNT = {NNT:.1f}")

# 視覺化
fig, ax = plt.subplots(figsize=(8, 5))
rates = [rate_t * 100, rate_p * 100]
bars = ax.bar(['治療組', '安慰劑組'], rates, color=['steelblue', 'salmon'], edgecolor='white', width=0.5)
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{rate:.1f}%',
            ha='center', fontsize=14, fontweight='bold')
ax.set_ylabel('達標率 (%)', fontsize=12)
ax.set_title(f'血糖達標率比較 (p = {p_chi:.4f})', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(rates) + 15)
plt.tight_layout()
plt.show()

## 6. 多組比較：ANOVA 劑量反應分析

假設試驗中有四組：安慰劑、低劑量、中劑量、高劑量。

In [ ]:
np.random.seed(2024)
n_per_dose = 50

dose_groups = {
    '安慰劑': np.random.normal(-0.5, 1.2, n_per_dose),
    '低劑量 (5mg)': np.random.normal(-1.2, 1.2, n_per_dose),
    '中劑量 (10mg)': np.random.normal(-2.0, 1.2, n_per_dose),
    '高劑量 (20mg)': np.random.normal(-2.8, 1.2, n_per_dose)
}

# One-way ANOVA
f_stat, p_anova = stats.f_oneway(*dose_groups.values())

print("=" * 50)
print("劑量反應分析：One-way ANOVA")
print("=" * 50)
for group, data in dose_groups.items():
    print(f"  {group:12s}: {data.mean():.2f} \u00b1 {data.std():.2f} mmol/L (n={len(data)})")
print(f"\nF = {f_stat:.3f}, p = {p_anova:.2e}")

# Post-hoc: Tukey HSD
from itertools import combinations
print(f"\nPost-hoc 比較（Bonferroni 校正, \u03b1/{len(list(combinations(dose_groups.keys(), 2)))}）:")
group_names = list(dose_groups.keys())
n_comparisons = len(list(combinations(group_names, 2)))
alpha_corrected = 0.05 / n_comparisons

for g1, g2 in combinations(group_names, 2):
    t_stat_ph, p_val_ph = stats.ttest_ind(dose_groups[g1], dose_groups[g2])
    sig = '***' if p_val_ph < alpha_corrected else 'ns'
    print(f"  {g1} vs {g2}: p = {p_val_ph:.4f} {sig}")

# 視覺化
fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot(dose_groups.values(), labels=dose_groups.keys(), patch_artist=True,
                boxprops=dict(facecolor='lightblue'), medianprops=dict(color='red', linewidth=2))
colors = ['#FF6B6B', '#FFA07A', '#87CEEB', '#4682B4']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('血糖變化量 (mmol/L)', fontsize=12)
ax.set_title(f'劑量反應分析 (ANOVA p = {p_anova:.2e})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. 非劣性檢定（TOST）

### 情境

若另一家藥廠推出仿製藥 GlucoDown-G，需要證明它「不比原廠藥差」。

**TOST（Two One-Sided Tests）**：
- 設定非劣性邊界 \u03b4（如 0.5 mmol/L）
- H\u2080: |\u03bc\u2081 - \u03bc\u2082| \u2265 \u03b4（新藥劣於原廠超過 \u03b4）
- H\u2081: |\u03bc\u2081 - \u03bc\u2082| < \u03b4（新藥不劣於原廠）
- 同時進行兩個單尾檢定，兩者都顯著才能宣稱非劣性

In [ ]:
np.random.seed(88)
n_each = 80

# 原廠藥效果
original_drug = np.random.normal(-2.0, 1.2, n_each)
# 仿製藥效果（略差一點，但在非劣性邊界內）
generic_drug = np.random.normal(-1.8, 1.2, n_each)

delta = 0.5  # 非劣性邊界

mean_diff_tost = generic_drug.mean() - original_drug.mean()
sp_tost = np.sqrt(((n_each-1)*original_drug.std(ddof=1)**2 + 
                    (n_each-1)*generic_drug.std(ddof=1)**2) / (2*n_each-2))
se_tost = sp_tost * np.sqrt(2/n_each)

# TOST: 兩個單尾檢定
# Test 1: H0: mu_generic - mu_original <= -delta  (差太多往左)
t1 = (mean_diff_tost + delta) / se_tost
p1 = 1 - stats.t.cdf(t1, df=2*n_each-2)

# Test 2: H0: mu_generic - mu_original >= delta   (差太多往右)
t2 = (mean_diff_tost - delta) / se_tost
p2 = stats.t.cdf(t2, df=2*n_each-2)

p_tost = max(p1, p2)

print("=" * 50)
print("非劣性檢定 (TOST)")
print("=" * 50)
print(f"\n原廠藥效果: {original_drug.mean():.3f} \u00b1 {original_drug.std():.3f} mmol/L")
print(f"仿製藥效果: {generic_drug.mean():.3f} \u00b1 {generic_drug.std():.3f} mmol/L")
print(f"差異: {mean_diff_tost:.3f} mmol/L")
print(f"非劣性邊界 \u03b4: {delta} mmol/L")
print(f"\nTest 1 (下界): t = {t1:.3f}, p = {p1:.4f}")
print(f"Test 2 (上界): t = {t2:.3f}, p = {p2:.4f}")
print(f"TOST p-value: {p_tost:.4f}")
print(f"\n結論: {'通過非劣性檢定' if p_tost < 0.05 else '未通過非劣性檢定'}")

# 視覺化
fig, ax = plt.subplots(figsize=(10, 4))
ci_tost = (mean_diff_tost - 1.96*se_tost, mean_diff_tost + 1.96*se_tost)
ax.errorbar(mean_diff_tost, 0, xerr=[[mean_diff_tost-ci_tost[0]], [ci_tost[1]-mean_diff_tost]],
            fmt='ko', markersize=10, capsize=8, linewidth=2)
ax.axvline(x=-delta, color='red', linestyle='--', linewidth=2, label=f'-\u03b4 = -{delta}')
ax.axvline(x=delta, color='red', linestyle='--', linewidth=2, label=f'+\u03b4 = {delta}')
ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
ax.fill_between([-delta, delta], -0.5, 0.5, alpha=0.1, color='green', label='非劣性區域')
ax.set_xlabel('仿製藥 - 原廠藥 差異 (mmol/L)', fontsize=12)
ax.set_title('非劣性檢定結果', fontsize=14, fontweight='bold')
ax.set_yticks([])
ax.set_ylim(-0.5, 0.5)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 8. 本章小結

| 分析 | 方法 | 使用場景 |
|------|------|----------|
| Table 1 | t 檢定 + 卡方 | 驗證隨機分配是否成功 |
| 主要指標 | 獨立樣本 t 檢定 | 兩組連續結果比較 |
| 前後比較 | 配對 t 檢定 | 同一組治療前後 |
| 達標率 | 卡方 + RR/OR/NNT | 二元結果比較 |
| 劑量反應 | ANOVA + post-hoc | 多組比較 |
| 非劣性 | TOST | 仿製藥/新療法「不劣於」對照 |

---

## 課堂練習

### 基礎題
1. 在上述 RCT 數據中，比較男性和女性的血糖變化量是否有差異。

### 進階題
2. 對治療組進行子群分析（subgroup analysis）：
   - BMI < 25 vs. BMI \u2265 25 的治療效果是否不同？
   - 提示：使用交互作用（interaction）分析

### 挑戰題
3. 實作一個 `clinical_report()` 函數，輸入 DataFrame，自動產生完整的臨床分析報告（Table 1 + 主要指標 + 次要指標）。

---

> **下一堂課**：[03_流行病學與公衛統計](./03_流行病學與公衛統計.ipynb) -- 風險指標、存活分析與流行病學方法